## **First stage evaluation**

### **LiquidAI**

In [1]:
from pathlib import Path
import json, os, re
from transformers import AutoProcessor, AutoModelForImageTextToText
from transformers.image_utils import load_image
import torch
from typing import defaultdict

/Users/elizabethgranda/Documents/measure_repo/MEASURE/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
resume_images_directory = "data/resumes_images"

In [ ]:
model_id = "LiquidAI/LFM2.5-VL-1.6B"
model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    device_map="cpu",
    dtype="bfloat16",
    trust_remote_code = True
)
processor = AutoProcessor.from_pretrained(model_id)

In [ ]:
liquidai_prompt = """Analyze the resume image and extract information into this exact JSON structure:

{
  "personal_info": {
    "full_name": "",
    "email": "",
    "phone": "",
    "location": "",
    "github": "",
    "linkedin": ""
  },
  "skills": [],
  "work_experience": [
    {
      "company": "",
      "role": "",
      "start_date": "",
      "end_date": "",
      "employment_description": ""
    }
  ],
  "education": [
    {
      "institution": "",
      "degree": "",
      "start_date": "",
      "end_date": "",
      "certifications": ""
    }
  ]
}

Only extract information that is clearly visible in the image. Do not add any explanatory text, only output the JSON.
If there is not information in the CV referring to one of the sections, leave the space in blank. 
Do not make up any information.
Do not translate or paraphrase the text."""


def extract_json_from_text(text):
    try:
        start = text.find('{')
        end = text.rfind('}') + 1
        if start != -1 and end != 0:
            json_str = text[start:end]
            return json.loads(json_str)
    except json.JSONDecodeError:
        pass
    
    try:
        start = text.find('[')
        end = text.rfind(']') + 1
        if start != -1 and end != 0:
            json_str = text[start:end]
            return json.loads(json_str)
    except json.JSONDecodeError:
        pass
    
    return {"raw_output": text, "error": "Could not parse JSON"}

In [ ]:
def process_single_resume(image_path, model, processor, output_dir="liquidai"):
    try:
        image = load_image(image_path)
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": liquidai_prompt},
                ],
            },
        ]
        
        inputs = processor(
            images=image,
            text=processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False),
            return_tensors="pt",
        ).to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=4096)
        
        result = processor.batch_decode(outputs, skip_special_tokens=True)[0]
        json_data = extract_json_from_text(result)
        image_name = Path(image_path).stem
        os.makedirs(output_dir, exist_ok=True)
        output_file = os.path.join(output_dir, f"{image_name}.json")
        
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(json_data, f, indent=2, ensure_ascii=False)
        
        print(f"Processed: {image_path}")
        print(f"Saved to: {output_file}")
        
        return {
            "image_path": image_path,
            "output_file": output_file,
            "success": True,
            "data": json_data
        }
        
    except Exception as e:
        print(f"Error processing {image_path}: {str(e)}")
        return {
            "image_path": image_path,
            "success": False,
            "error": str(e)
        }

In [ ]:
def process_resume_directory(directory_path, output_dir="/output_dir", model="LiquidAI/LFM2.5-VL-1.6B"):
    image_extensions = {'.jpg', '.jpeg', '.png'}
    
    image_files = []
    for file in os.listdir(directory_path):
        file_path = os.path.join(directory_path, file)
        if os.path.isfile(file_path) and Path(file).suffix.lower() in image_extensions:
            image_files.append(file_path)

    model_id = model
    model = AutoModelForImageTextToText.from_pretrained(
        model_id,
        device_map="cpu",
        torch_dtype=torch.bfloat16
    )
    processor = AutoProcessor.from_pretrained(model_id)
    results = []
    for i, image_path in enumerate(image_files, 1):
        print(f"Processing {i}/{len(image_files)}: {Path(image_path).name}")
        result = process_single_resume(image_path, model, processor, output_dir)
        results.append(result)
    
    return results

In [ ]:
results = process_resume_directory(resume_images_directory)

## **Second stage evaluation**



In [ ]:
prompt_w_context_between_pages_liquidai = """You are given multiple images that together form a single resume (CV).
The pages are provided in order and must be interpreted as ONE document.

Extract structured information from the COMPLETE resume using ALL pages.
Information may appear on any page.

Return ONLY valid JSON following the provided schema:
{
  "personal_info": {
    "full_name": "",
    "email": "",
    "phone": "",
    "location": "",
    "github": "",
    "linkedin": ""
  },
  "skills": [],
  "work_experience": [
    {
      "company": "",
      "role": "",
      "start_date": "",
      "end_date": "",
      "employment_description": ""
    }
  ],
  "education": [
    {
      "institution": "",
      "degree": "",
      "start_date": "",
      "end_date": "",
      "certifications": ""
    }
  ]
}


Rules:
- Do NOT translate, rewrite, or summarize text.
- Preserve the original language exactly as shown.
- Do NOT hallucinate or infer missing information.
- Do NOT duplicate entries across pages.
- If a section continues across pages, merge it correctly.
The output must represent the full resume as a single entity.
"""

def process_multi_page_resume_liquidai(cv_name, image_paths, processor, output_dir="/output_dir", 
                                       model = model, max_new_tokens=8192):
    try:
        images = [load_image(str(p)) for p in image_paths]
        messages = [
            {
                "role": "user",
                "content": (
                    [{"type": "image", "image": img} for img in images]
                    + [{"type": "text", "text": prompt_w_context_between_pages_liquidai}]
                ),
            }
        ]
        prompt = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False,)
        inputs = processor(images=images, text=prompt, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
            )

        decoded = processor.batch_decode(outputs, skip_special_tokens=True)[0]
        json_data = extract_json_from_text(decoded)
        os.makedirs(output_dir, exist_ok=True)
        output_file = os.path.join(output_dir, f"{cv_name}.json")

        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(json_data, f, indent=2, ensure_ascii=False)

        print(f"Saved CV JSON: {output_file}")

        return {
            "cv_name": cv_name,
            "success": True,
            "output_file": output_file,
            "data": json_data,
        }

    except Exception as e:
        print(f"Error processing CV {cv_name}: {e}")
        return {
            "cv_name": cv_name,
            "success": False,
            "error": str(e),
        }


In [ ]:
def group_resumes_from_directory(image_dir):
    
    image_dir = Path(image_dir)
    image_paths = sorted(
        p for p in image_dir.iterdir()
        if p.suffix.lower() in [".png", ".jpg", ".jpeg"]
    )

    groups = defaultdict(list)
    pattern = re.compile(r"(.*)_page_\d+", re.IGNORECASE)

    for p in image_paths:
        match = pattern.match(p.stem)
        if not match:
            continue

        cv_id = match.group(1)
        groups[cv_id].append(p)

    for cv_id in groups:
        groups[cv_id] = sorted(groups[cv_id])

    return dict(groups)

In [ ]:
def process_resume_directory_liquidai(image_dir, output_dir, model_id=model_id):

    image_dir = Path(image_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    grouped_resumes = group_resumes_from_directory(image_dir)

    print(f"Found {len(grouped_resumes)} CVs\n")

    model = AutoModelForImageTextToText.from_pretrained(
        model_id,
        device_map="cpu",  
        torch_dtype=torch.bfloat16,
    )
    processor = AutoProcessor.from_pretrained(model_id)

    results = []

    for cv_name, image_paths in grouped_resumes.items():
        print(f"Processing CV: {cv_name}")
        for i, p in enumerate(image_paths, 1):
            print(f"Page {i}: {p.name}")

        result = process_multi_page_resume_liquidai(
            cv_name=cv_name,
            image_paths=image_paths,
            model=model,
            processor=processor,
            output_dir=output_dir,
        )

        results.append(result)

    print("ALL DONE")
    return results

In [ ]:
results = process_resume_directory_liquidai(resume_images_directory, output_dir)